In [18]:
import mitsuba as mi
import drjit as dr

print("Mitsuba version:", mi.__version__)
print("Dr.Jit version:", dr.__version__)
print("Current variant:", mi.variant())

Mitsuba version: 3.9.1
Dr.Jit version: 1.5.0
Current variant: cuda_ad_mono_polarized


In [19]:
mi.set_variant("cuda_ad_mono_polarized")
print("CUDA variant:", mi.variant())

CUDA variant: cuda_ad_mono_polarized


In [20]:
# Imported Sionna and the functions we will use
# Imported the NumPy library for mathematical operations
import sionna.rt
import mitsuba as mi
from sionna.rt import (
    PlanarArray,
    Transmitter,
    Receiver,
    load_scene,
    PathSolver,
    SceneObject,
    ITURadioMaterial
    )
import numpy as np
import matplotlib.pyplot as plt

In [21]:
# Called our environment and loaded the scene
scene = load_scene()

In [22]:
# Set our frequency to 3.5 GHz and change it to 7 GHz
#and the other parameters
scene.frequency = 7e9
tx_power_dbm = 30
c = 299792458.0
r = 20

In [23]:
# Using the PlanarArray function, we defined the number of Tx antenna elements, to be used as our base station, antenna properties and orientation

scene.tx_array= PlanarArray(
    num_rows=1, 
    num_cols=1,
    pattern="iso", 
    polarization="V",)

In [24]:
# We defined the properties of our receiving antenna
scene.rx_array= PlanarArray(
    num_rows=1,
    num_cols=1,
    pattern="iso",
    polarization="V",
)

In [25]:
# Named our transmitter Base Station 1 and defined its position
bs = Transmitter(
    name="BS1",
    position=[0, 0, 0]
)

In [26]:
# Added our base station to the scene
scene.add(bs)

In [27]:
# Named our first receiver and set its position at a distance of r from the base station
rx1 = Receiver(
    name="RX1",
    position=[r, 0, 0]
)

In [28]:
# Named our second receiver and set its position at a distance of 2*r from the base station# Named our second receiver and set its position at a distance of 2*r from the base station
rx2 = Receiver(
    name="RX2",
    position=[2*r, 0, 0]
)

In [29]:
# Named our third receiver and set its position at a distance of 3*r from the base station
rx3 = Receiver(
    name="RX3",
    position=[3*r, 0, 0]
)

In [30]:
# Named our fourth receiver and set its position at a distance of 10*r from the base station
rx4 = Receiver(
    name="RX4",
    position=[10*r, 0, 0]
)

In [31]:
# Added our receivers to the scene
scene.add(rx1)
scene.add(rx2)
scene.add(rx3)
scene.add(rx4)

In [32]:
# Printed the position of each receiver to make sure their positions were correct
for name in ["RX1", "RX2", "RX3", "RX4"]:
    rx = scene.get(name)
    print(name, np.array(rx.position))

RX1 [[20.]
 [ 0.]
 [ 0.]]
RX2 [[40.]
 [ 0.]
 [ 0.]]
RX3 [[60.]
 [ 0.]
 [ 0.]]
RX4 [[200.]
 [  0.]
 [  0.]]


In [33]:
# Defined the constant speed of light, wavelength, and base station power
# Calculated the free space path loss values for each receiver
# path gain = - path loss
# Calculated the received signal strength (RSS) for each receiver
c= 299792458.0
wavelength = c/ scene.frequency
bs_power_dbm = 30
fspl1 = 20*np.log10(4*np.pi*rx1.position[0]/wavelength)
fspl2 = 20*np.log10(4*np.pi*rx2.position[0]/wavelength)
fspl3 = 20*np.log10(4*np.pi*rx3.position[0]/wavelength)
fspl4 = 20*np.log10(4*np.pi*rx4.position[0]/wavelength)

rss1 = bs_power_dbm - fspl1
rss2 = bs_power_dbm - fspl2
rss3 = bs_power_dbm - fspl3
rss4 = bs_power_dbm - fspl4

print(f"RSS at RX1: {rss1} dBm")
print(f"RSS at RX2: {rss2} dBm")
print(f"RSS at RX3: {rss3} dBm")
print(f"RSS at RX4: {rss4} dBm")

print(f"FSPL at RX1: {fspl1} dB")
print(f"FSPL at RX2: {fspl2} dB")
print(f"FSPL at RX3: {fspl3} dB")
print(f"FSPL at RX4: {fspl4} dB")

# Convert received power from dBm to nanowatts for comparing
p_rx1 = 10**((rss1 - 30) / 10) * 1e9
p_rx2 = 10**((rss2 - 30) / 10) * 1e9
p_rx3 = 10**((rss3 - 30) / 10) * 1e9
p_rx4 = 10**((rss4 - 30) / 10) * 1e9

print(f"RX1: {p_rx1.item():.2f} nW")
print(f"RX2: {p_rx2.item():.2f} nW")
print(f"RX3: {p_rx3.item():.2f} nW")
print(f"RX4: {p_rx4.item():.2f} nW")

print(f"RX2 / RX1: {((p_rx2 / p_rx1).item()):.2f}")

RSS at RX1: [-45.370346] dBm
RSS at RX2: [-51.390945] dBm
RSS at RX3: [-54.912766] dBm
RSS at RX4: [-65.37034] dBm
FSPL at RX1: [75.370346] dB
FSPL at RX2: [81.390945] dB
FSPL at RX3: [84.912766] dB
FSPL at RX4: [95.37034] dB
RX1: 29.04 nW
RX2: 7.26 nW
RX3: 3.23 nW
RX4: 0.29 nW
RX2 / RX1: 0.25


In [34]:
# Receiver distances (meters)
distances = [r, 2*r, 3*r, 10*r]

# Received signal strength values (dBm)
rss_values = [
    float(rss1),
    float(rss2),
    float(rss3),
    float(rss4)
]

# Plot RSS vs Distance
plt.figure(figsize=(8, 5))

plt.plot(distances, rss_values, marker="o", linestyle="-")

# Add receiver labels
for i in range(4):
    plt.annotate(
        f"RX{i+1}",
        (distances[i], rss_values[i]),
        xytext=(5, 8),
        textcoords="offset points"
    )

plt.xlabel("Distance (m)")
plt.ylabel("RSS (dBm)")
plt.title("Received Signal Strength vs Distance")
plt.grid(True)

plt.show()

TypeError: only 0-dimensional arrays can be converted to Python scalars

In [ ]:
# Create a building geometry using Mitsuba
building_mesh = mi.load_dict({
    "type": "cube",
    "to_world": mi.ScalarTransform4f()
        .translate([30, 0, -4])
        .scale([5, 5, 7.5])
})

print(type(building_mesh))

In [ ]:
# Create a Sionna scene object from the Mitsuba mesh
building1 = SceneObject(
    mi_mesh=building_mesh,
    name="building1",
    radio_material=ITURadioMaterial(
        name="concrete",
        itu_type="concrete"
    )
)

In [ ]:
# Add the building to the scene
scene.add(building1)

In [ ]:
scene.preview()

In [ ]:
solver = PathSolver()

paths_with = solver(
    scene,
    max_depth=3,
    los=True,
    specular_reflection=True,
    diffuse_reflection=False,
    diffraction= False
)

In [ ]:
# Remove the building
scene.remove("building1")

# Calculate propagation paths without the building
paths_without = solver(
    scene,
    max_depth=3,
    los=True,
    specular_reflection=True,
    diffuse_reflection=False,
    diffraction=False
)

In [1]:
#obje kapatırsa  rss ve path gain değerleri değişimi
# Get channel coefficients

# Calculate path gain and RSS from channel coefficients
def calculate_rss(paths, tx_power_dbm=30):

    a_real, a_imag = paths.a
    a = np.array(a_real) + 1j * np.array(a_imag)

    # Combine propagation paths coherently
    h = np.sum(a, axis=-1)

    # Calculate path gain
    path_gain = np.abs(h)**2
    path_gain_db = 10 * np.log10(path_gain)

    # Calculate received signal strength
    rss_dbm = tx_power_dbm + path_gain_db

    return path_gain_db.reshape(-1), rss_dbm.reshape(-1)

In [2]:
# With building
gain_with, rss_with = calculate_rss(paths_with)

# Without building
gain_without, rss_without = calculate_rss(paths_without)

# Compare results
for i in range(4):
    print(f"RX{i+1}")
    print(f"RSS without building: {rss_without[i]:.2f} dBm")
    print(f"RSS with building:    {rss_with[i]:.2f} dBm")
    print(f"RSS difference:       {rss_with[i] - rss_without[i]:.2f} dB")
    print()

NameError: name 'paths_with' is not defined

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(distances, rss_without, marker="o", label="Without Building")
plt.plot(distances, rss_with, marker="s", label="With Building")

plt.xlabel("Distance (m)")
plt.ylabel("RSS (dBm)")
plt.title("Effect of Building on Received Signal Strength")
plt.legend()
plt.grid(True)
plt.show()

Research Question 3: What Happens When Only LoS Is Considered?

In [ ]:
# Add the concrete building back to the scene
scene.add(building1)

paths_los = solver(
    scene,
    max_depth=0,
    los=True,
    specular_reflection=False,
    diffuse_reflection=False,
    diffraction=False
)

gain_los, rss_los = calculate_rss(paths_los)

print(f"RX1 RSS (LoS only): {rss_los[0]:.2f} dBm")

Research Question 4: How Does Diffraction Affect RSS?

In [ ]:
# Calculate paths with diffraction enabled
paths_diff = solver(
    scene,
    max_depth=3,
    los=True,
    specular_reflection=False,
    diffuse_reflection=False,
    diffraction=True
)

# Calculate paths with diffraction disabled
paths_no_diff = solver(
    scene,
    max_depth=3,
    los=True,
    specular_reflection=False,
    diffuse_reflection=False,
    diffraction=False
)

# Calculate RSS
gain_diff, rss_diff = calculate_rss(paths_diff)
gain_no_diff, rss_no_diff = calculate_rss(paths_no_diff)

# Compare
for i in range(4):
    print(f"RX{i+1}")
    print(f"With diffraction:    {rss_diff[i]:.6f} dBm")
    print(f"Without diffraction: {rss_no_diff[i]:.6f} dBm")
    print()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(distances, rss_diff, marker="o", label="With Diffraction")
plt.plot(distances, rss_no_diff, marker="s", label="Without Diffraction")

plt.xlabel("Distance (m)")
plt.ylabel("RSS (dBm)")
plt.title("Effect of Diffraction on RSS")
plt.legend()
plt.grid(True)
plt.show()

Research Question 5: Which Materials Affect Signal Propagation More?

#if we make this code block we can choose the material specifically
# Remove the concrete building from the scene
scene.remove("building3")
# Select the material manually
selected_material = "concrete"  # concrete, glass, brick, wood, metal

# Remove the previous building if it exists
if scene.get("building3") is not None:
    scene.remove("building3")

# Add building to scene
scene.add(building3)

# Calculate propagation paths
paths_material = solver(
    scene,
    max_depth=3,
    los=True,
    specular_reflection=True,
    diffuse_reflection=False,
    diffraction=False
)

# Calculate RSS and path gain
gain_material, rss_material = calculate_rss(paths_material)

# Print results
print(f"\nMaterial: {selected_material.upper()}")

for i in range(4):
    print(f"RX{i+1}")
    print(f"  RSS:       {rss_material[i]:.2f} dBm")
    print(f"  Path Gain: {gain_material[i]:.2f} dB")

In [ ]:
# Research Question 5: Which materials affect signal propagation more?

# Remove the concrete building from the scene
scene.remove("building1")

# Materials to compare
materials = ["concrete", "glass", "brick", "wood"]
material_results = {}

for material_name in materials:

    # Create a building with the selected material
    building3 = SceneObject(
        mi_mesh=building_mesh,
        name="building3",
        radio_material=ITURadioMaterial(
            name=f"material_{material_name}",
            itu_type=material_name
        )
    )

    scene.add(building3)

    # Calculate propagation paths
    paths = solver(
        scene,
        max_depth=3,
        los=True,
        specular_reflection=True,
        diffuse_reflection=False,
        diffraction=False
    )

    # Calculate RSS and path gain
    gain, rss = calculate_rss(paths)

    material_results[material_name] = {
        "gain": gain.copy(),
        "rss": rss.copy()
    }

    # Print results
    print(f"\nMaterial: {material_name.upper()}")

    for i in range(4):
        print(f"RX{i+1}")
        print(f"  RSS:       {rss[i]:.2f} dBm")
        print(f"  Path Gain: {gain[i]:.2f} dB")

    # Remove building before testing the next material
    scene.remove("building3")

In [ ]:
#Material Comparison
plt.figure(figsize=(9, 5))

for material_name, results in material_results.items():
    plt.plot(
        distances,
        results["rss"],
        marker="o",
        label=material_name.capitalize()
    )

plt.xlabel("Distance (m)")
plt.ylabel("RSS (dBm)")
plt.title("Effect of Building Materials on RSS")
plt.legend(title="Building Material")
plt.grid(True)
plt.show()

In [3]:
# path gain comparison
plt.figure(figsize=(9, 5))

for material_name, results in material_results.items():
    plt.plot(
        distances,
        results["gain"],
        marker="o",
        label=material_name.capitalize()
    )

plt.xlabel("Distance (m)")
plt.ylabel("Path Gain (dB)")
plt.title("Effect of Building Materials on Path Gain")
plt.legend(title="Building Material")
plt.grid(True)
plt.show()

NameError: name 'plt' is not defined